# FINAL v7.1 - Direct Yahoo ETF universe (audit fix)

This notebook replaces the fixed 29-ticker research pool.

It uses three Alpha Vantage listing snapshots to create a broader lifecycle-aware ETF universe:

- ETFs active immediately before the first August 2016 holding month;
- ETFs active at the end of the study;
- ETFs delisted by the end of the study.

Listing and delisting dates determine whether an ETF may enter each monthly ranking. Product-name rules classify ETFs into the seven sleeves without using return performance. Yahoo daily data then determine the top two or three ETFs by trailing average daily share volume at every month-end. SOXX remains fixed.

This materially reduces the current-survivor restriction, but it is not identical to a licensed survivor-bias-free database. Historical names, category changes, ticker changes and unavailable Yahoo histories remain auditable limitations.
The Yahoo downloader is rate-limit aware, resumable and cache preserving.

This version does not import or call yfinance. It requests Yahoo's chart JSON directly.

This revision fixes the Pandas index-alignment error in the future-data audit.


In [ ]:
%pip -q install pandas numpy openpyxl requests


In [ ]:
from __future__ import annotations

import getpass
import hashlib
import io
import json
import os
import re
import random
import time
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display


PROJECT_DIR = Path.cwd()
ALPHA_DIR = PROJECT_DIR / "data" / "raw" / "alpha_vantage"
CSV_DIR = PROJECT_DIR / "yahoo_csv"
RESULTS_ROOT = PROJECT_DIR / "dynamic_monthly_universe_results"
for folder in (ALPHA_DIR, CSV_DIR, RESULTS_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

FIRST_HOLDING_MONTH = pd.Period("2016-08", freq="M")
LAST_HOLDING_MONTH = pd.Period("2026-08", freq="M")
START_SNAPSHOT_DATE = pd.Timestamp("2016-07-29")
END_SNAPSHOT_DATE = pd.Timestamp("2026-07-31")

LIQUIDITY_LOOKBACK_DAYS = 63
MIN_LIQUIDITY_DAYS = 40
MIN_AVG_DAILY_SHARE_VOLUME = 0  # Ranking only; no unrequested absolute volume cutoff
RANKING_METRIC = "avg_daily_share_volume"
TARGET_PER_SLEEVE = 3
MIN_PER_SLEEVE = 2

DOWNLOAD_WITH_YAHOO_CHART = True
FORCE_ALPHA_DOWNLOAD = False
FORCE_YAHOO_DOWNLOAD = False
YAHOO_START = "2011-01-01"
YAHOO_END_EXCLUSIVE = "2026-08-06"

# Rate-safe Yahoo controls. Successful CSVs are always preserved.
YAHOO_INTER_REQUEST_DELAY_SECONDS = 3.0
YAHOO_MAX_RETRIES_PER_SYMBOL = 1
YAHOO_COOLDOWN_SECONDS = 0
YAHOO_STOP_AFTER_CONSECUTIVE_FAILURES = 3

FIXED_SYMBOL = "SOXX"
BENCHMARK_SYMBOL = "SPY"

# Yahoo does not expose histories for these delisted/reused tickers.
KNOWN_YAHOO_UNAVAILABLE = {"BIGD", "GNRX", "OCTA"}
ALLOWED_EXCHANGES = {"NYSE", "NYSE ARCA", "NASDAQ", "BATS", "CBOE", "NYSE MKT"}

API_KEY = os.environ.get("ALPHAVANTAGE_API_KEY") or getpass.getpass(
    "Alpha Vantage API key (hidden; not saved): "
)
if not API_KEY.strip():
    raise ValueError("An Alpha Vantage API key is required.")

print(f"Project folder: {PROJECT_DIR}")
print(f"Holding months: {FIRST_HOLDING_MONTH} to {LAST_HOLDING_MONTH}")
print("Every holding month is ranked separately using only the preceding month-end data.")

## Build a broader lifecycle-aware ETF research universe

The discovery rule uses listing status and product names, never subsequent returns. It intentionally retains the full mechanically discovered pool before the monthly liquidity ranking.

The three Alpha Vantage calls are cached, so rerunning the notebook does not consume additional calls unless `FORCE_ALPHA_DOWNLOAD = True`.

In [ ]:
ALPHA_URL = "https://www.alphavantage.co/query"

SLEEVE_RULES = [
    ("Pharmaceuticals", r"\b(pharmaceuticals?|pharma|generic drugs?)\b"),
    ("Consumer Staples", r"consumer staples|food\s*&\s*beverage"),
    ("Capital Markets & Exchanges", r"broker[- ]?dealers?|capital markets|securities exchanges?"),
    ("Aerospace & Defense", r"aerospace|defen[cs]e"),
    ("Insurance", r"insurance|property\s*(?:&|and)?\s*casualty"),
    ("Utilities", r"\butilities?\b"),
    ("Energy", r"\benergy\b|oil\s*&\s*gas|oil services|natural gas"),
]
COMMON_EXCLUSION = re.compile(
    r"\b(?:2x|3x|ultra|ultrashort|ultrashrt|inverse|short|bear|bull|leveraged|etn|income)\b|high yield bond|resources trust$|fund inc$|development company$|total return fund$",
    flags=re.IGNORECASE,
)
FOREIGN_EXCLUSION = re.compile(
    r"\b(?:global|international|world|china|canadian|asia|europe|korea|ex[- ]?u\.?s\.?|emerging markets?)\b",
    flags=re.IGNORECASE,
)
NON_EQUITY_ENERGY_EXCLUSION = re.compile(
    r"crude oil|brent|gasoline|heating oil|commodity|futures?|natural gas fund|mlp|midstream|infrastructure|clean.*energy|green.*energy|renewable|solar|wind|low carbon|energy transition|energy storage|progressive energy|smart energy|sustainable energy|ex[- ]?energy|db energy fund",
    flags=re.IGNORECASE,
)


def alpha_message(text: str) -> None:
    stripped = text.lstrip()
    if not stripped.startswith("{"):
        return
    payload = json.loads(stripped)
    message = payload.get("Error Message") or payload.get("Information") or payload.get("Note") or str(payload)
    raise RuntimeError(message)


def listing_status(state: str, as_of_date: pd.Timestamp) -> pd.DataFrame:
    cache = ALPHA_DIR / f"listing_status_{state}_{as_of_date:%Y-%m-%d}.csv"
    if cache.exists() and not FORCE_ALPHA_DOWNLOAD:
        print(f"CACHED: {cache.name}")
        return pd.read_csv(cache)
    response = requests.get(
        ALPHA_URL,
        params={
            "function": "LISTING_STATUS",
            "date": as_of_date.strftime("%Y-%m-%d"),
            "state": state,
            "apikey": API_KEY,
        },
        timeout=90,
    )
    response.raise_for_status()
    alpha_message(response.text)
    frame = pd.read_csv(io.StringIO(response.text))
    if frame.empty or "symbol" not in frame.columns:
        raise RuntimeError(f"Unexpected LISTING_STATUS response: {frame.columns.tolist()}")
    frame.to_csv(cache, index=False)
    print(f"SAVED: {cache.name} ({len(frame):,} rows)")
    return frame


def normalize_listing(frame: pd.DataFrame, snapshot: str) -> pd.DataFrame:
    output = frame.copy()
    output.columns = [str(column).strip() for column in output.columns]
    for column in ["symbol", "name", "exchange", "assetType", "status"]:
        if column not in output.columns:
            output[column] = ""
        output[column] = output[column].astype(str).str.strip()
    for column in ["ipoDate", "delistingDate"]:
        if column not in output.columns:
            output[column] = pd.NaT
        output[column] = pd.to_datetime(output[column], errors="coerce")
    output["snapshot"] = snapshot
    return output


start_active = normalize_listing(listing_status("active", START_SNAPSHOT_DATE), "START_ACTIVE")
end_active = normalize_listing(listing_status("active", END_SNAPSHOT_DATE), "END_ACTIVE")
end_delisted = normalize_listing(listing_status("delisted", END_SNAPSHOT_DATE), "END_DELISTED")

listing_union_raw = pd.concat([start_active, end_active, end_delisted], ignore_index=True)
listing_union_raw = listing_union_raw.loc[
    listing_union_raw["assetType"].str.upper().eq("ETF")
].copy()
listing_union_raw["symbol"] = listing_union_raw["symbol"].str.upper()

# Prefer the historical start-snapshot name when a ticker already existed in 2016.
snapshot_priority = {"START_ACTIVE": 0, "END_DELISTED": 1, "END_ACTIVE": 2}
listing_union_raw["snapshot_priority"] = listing_union_raw["snapshot"].map(snapshot_priority)
listing_union_raw = listing_union_raw.sort_values(["symbol", "snapshot_priority"])

lifecycle_rows = []
for symbol, group in listing_union_raw.groupby("symbol", sort=True):
    preferred = group.iloc[0]
    ipo = group["ipoDate"].dropna().min() if group["ipoDate"].notna().any() else pd.NaT
    delist = group["delistingDate"].dropna().max() if group["delistingDate"].notna().any() else pd.NaT
    lifecycle_rows.append({
        "symbol": symbol,
        "name": preferred["name"],
        "exchange": preferred["exchange"],
        "ipoDate": ipo,
        "delistingDate": delist,
        "seen_start_active": bool(group["snapshot"].eq("START_ACTIVE").any()),
        "seen_end_active": bool(group["snapshot"].eq("END_ACTIVE").any()),
        "seen_end_delisted": bool(group["snapshot"].eq("END_DELISTED").any()),
        "snapshot_records": int(len(group)),
    })
lifecycle = pd.DataFrame(lifecycle_rows)

allowed = {exchange.upper() for exchange in ALLOWED_EXCHANGES}
lifecycle["allowed_exchange"] = lifecycle["exchange"].str.upper().isin(allowed)
lifecycle["common_exclusion"] = lifecycle["name"].str.contains(COMMON_EXCLUSION, na=False)
lifecycle["foreign_exclusion"] = lifecycle["name"].str.contains(FOREIGN_EXCLUSION, na=False)

def classify_name(name: str) -> tuple[str, str]:
    matches = [(sleeve, pattern) for sleeve, pattern in SLEEVE_RULES if re.search(pattern, str(name), flags=re.IGNORECASE)]
    if not matches:
        return "", "NO_SLEEVE_KEYWORD"
    sleeve, pattern = matches[0]
    if sleeve == "Energy" and NON_EQUITY_ENERGY_EXCLUSION.search(str(name)):
        return "", "NON_CORE_OR_NON_EQUITY_ENERGY"
    return sleeve, pattern

classifications = lifecycle["name"].map(classify_name)
lifecycle["sleeve"] = classifications.map(lambda value: value[0])
lifecycle["matched_rule"] = classifications.map(lambda value: value[1])
lifecycle["mechanically_included"] = (
    lifecycle["allowed_exchange"]
    & ~lifecycle["common_exclusion"]
    & ~lifecycle["foreign_exclusion"]
    & lifecycle["sleeve"].ne("")
    & lifecycle["ipoDate"].notna()
    & lifecycle["ipoDate"].le(END_SNAPSHOT_DATE)
    & (lifecycle["delistingDate"].isna() | lifecycle["delistingDate"].ge(START_SNAPSHOT_DATE))
)

research_pool = lifecycle.loc[lifecycle["mechanically_included"]].copy()
research_pool["classification_review"] = "REVIEW_PRODUCT_OBJECTIVE_WITH_HISTORICAL_SOURCE"
research_pool = research_pool.sort_values(["sleeve", "symbol"]).reset_index(drop=True)
if research_pool.groupby("sleeve")["symbol"].nunique().lt(MIN_PER_SLEEVE).any():
    counts = research_pool.groupby("sleeve")["symbol"].nunique().to_dict()
    raise ValueError(f"A sleeve has fewer than two discovered ETFs: {counts}")

required_symbols = sorted(set(research_pool["symbol"]) | {FIXED_SYMBOL, BENCHMARK_SYMBOL})
download_checklist = pd.DataFrame({
    "symbol": required_symbols,
    "purpose": [
        "Fixed holding" if symbol == FIXED_SYMBOL else
        "Benchmark" if symbol == BENCHMARK_SYMBOL else
        "Dynamically discovered monthly research pool"
        for symbol in required_symbols
    ],
    "accepted_filename": [f"{symbol}.csv" for symbol in required_symbols],
})

print(f"ETF lifecycle records before sector rules: {len(lifecycle):,}")
print(f"Mechanically discovered sector ETF candidates: {len(research_pool):,}")
print(f"Yahoo symbols requested including SOXX and SPY: {len(required_symbols):,}")
display(research_pool.groupby("sleeve")["symbol"].apply(list).to_frame("discovered_symbols"))

## Download Yahoo daily CSVs without yfinance

This cell calls Yahoo's chart JSON endpoint directly. Each ticker uses one HTTP request and every successful response is converted into a conventional CSV immediately. Existing CSVs remain cached, so rerunning resumes from the first missing ticker.

SOXX and SPY are requested first. A genuinely unavailable or delisted ticker is recorded as missing rather than converted into zero volume.


In [ ]:
if DOWNLOAD_WITH_YAHOO_CHART:
    from urllib.parse import quote

    def utc_epoch(date_string: str) -> int:
        return int(pd.Timestamp(date_string, tz="UTC").timestamp())


    def direct_yahoo_history(symbol: str, session: requests.Session) -> pd.DataFrame:
        period1 = utc_epoch(YAHOO_START)
        period2 = utc_epoch(YAHOO_END_EXCLUSIVE)
        encoded_symbol = quote(symbol, safe="")
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{encoded_symbol}"
        response = session.get(
            url,
            params={
                "period1": period1,
                "period2": period2,
                "interval": "1d",
                "events": "history",
                "includeAdjustedClose": "true",
            },
            timeout=60,
        )
        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After", "")
            raise RuntimeError(f"HTTP 429 rate limit; Retry-After={retry_after}")
        response.raise_for_status()
        payload = response.json()
        chart = payload.get("chart", {})
        if chart.get("error"):
            raise RuntimeError(str(chart["error"]))
        results = chart.get("result") or []
        if not results:
            raise RuntimeError("Yahoo chart returned no result")
        result = results[0]
        timestamps = result.get("timestamp") or []
        quotes = (result.get("indicators", {}).get("quote") or [{}])[0]
        adjusted_blocks = result.get("indicators", {}).get("adjclose") or []
        adjusted = adjusted_blocks[0].get("adjclose", []) if adjusted_blocks else []
        if not timestamps:
            raise RuntimeError("Yahoo chart returned no observations")

        size = len(timestamps)
        def values(name: str):
            data = quotes.get(name) or []
            return list(data) + [None] * max(0, size - len(data))

        adjusted_values = list(adjusted) + [None] * max(0, size - len(adjusted))
        frame = pd.DataFrame({
            "Date": pd.to_datetime(timestamps, unit="s", utc=True).tz_convert(None).normalize(),
            "Open": values("open")[:size],
            "High": values("high")[:size],
            "Low": values("low")[:size],
            "Close": values("close")[:size],
            "Adj Close": adjusted_values[:size],
            "Volume": values("volume")[:size],
        })
        if not adjusted_blocks:
            frame["Adj Close"] = frame["Close"]
        frame = frame.dropna(subset=["Date", "Close", "Volume"])
        frame = frame.loc[(frame["Close"] > 0) & (frame["Volume"] >= 0)]
        frame = frame.sort_values("Date").drop_duplicates("Date", keep="last")
        if frame.empty:
            raise RuntimeError("Yahoo chart had no usable Close/Volume rows")
        return frame


    candidate_order = sorted(set(required_symbols) - {FIXED_SYMBOL, BENCHMARK_SYMBOL})
    download_order = [FIXED_SYMBOL, BENCHMARK_SYMBOL] + candidate_order
    download_rows = []
    consecutive_failures = 0
    stopped_early = False
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
        "Accept": "application/json,text/plain,*/*",
    })

    for position, symbol in enumerate(download_order, start=1):
        destination = CSV_DIR / f"{symbol}.csv"
        if symbol in KNOWN_YAHOO_UNAVAILABLE and not destination.exists():
            download_rows.append({
                "symbol": symbol, "status": "KNOWN_UNAVAILABLE", "file": str(destination),
                "error": "Delisted/reused ticker history unavailable from Yahoo",
            })
            print(f"[{position}/{len(download_order)}] KNOWN_UNAVAILABLE: {symbol}")
            consecutive_failures = 0
            continue
        if destination.exists() and not FORCE_YAHOO_DOWNLOAD:
            download_rows.append({
                "symbol": symbol, "status": "CACHED", "file": str(destination), "error": "",
            })
            print(f"[{position}/{len(download_order)}] CACHED: {symbol}")
            consecutive_failures = 0
            continue

        time.sleep(YAHOO_INTER_REQUEST_DELAY_SECONDS + random.uniform(0.0, 0.8))
        try:
            frame = direct_yahoo_history(symbol, session)
            frame.to_csv(destination, index=False)
            download_rows.append({
                "symbol": symbol, "status": "OK", "file": str(destination),
                "rows": len(frame), "error": "",
            })
            consecutive_failures = 0
            print(f"[{position}/{len(download_order)}] OK: {symbol} ({len(frame):,} rows)")
        except Exception as exc:
            consecutive_failures += 1
            download_rows.append({
                "symbol": symbol, "status": "FAILED", "file": str(destination), "error": str(exc),
            })
            print(f"[{position}/{len(download_order)}] FAILED: {symbol}: {exc}")
            if consecutive_failures >= YAHOO_STOP_AFTER_CONSECUTIVE_FAILURES:
                stopped_early = True
                print("Direct Yahoo stopped after consecutive failures. Cached CSVs are preserved.")
                break

    attempted = {row["symbol"] for row in download_rows}
    for symbol in download_order:
        if symbol not in attempted:
            download_rows.append({
                "symbol": symbol, "status": "NOT_ATTEMPTED_THIS_RUN",
                "file": str(CSV_DIR / f"{symbol}.csv"),
                "error": "Stopped after consecutive failures",
            })

    yahoo_download_log = pd.DataFrame(download_rows)
    yahoo_download_log.to_csv(PROJECT_DIR / "latest_direct_yahoo_download_log.csv", index=False)
    display(yahoo_download_log["status"].value_counts().rename("count").to_frame())
    remaining = yahoo_download_log.loc[
        ~yahoo_download_log["status"].isin(["OK", "CACHED", "KNOWN_UNAVAILABLE"]), "symbol"
    ].tolist()
    print(f"Remaining files: {len(remaining)}")
    if remaining:
        print("First remaining symbols:", remaining[:20])
        print("Rerun this cell later to resume; do not delete yahoo_csv.")
    elif not stopped_early:
        print("Direct Yahoo CSV set is complete.")
else:
    print("Direct Yahoo downloading is disabled. Upload or copy CSV files into yahoo_csv.")


In [ ]:
# Optional Colab upload fallback. Skip when automatic Yahoo download succeeds.
try:
    from google.colab import files
    uploaded = files.upload()
    for filename, content in uploaded.items():
        destination = CSV_DIR / Path(filename).name
        destination.write_bytes(content)
        print(f"Saved: {destination}")
except ImportError:
    print("Not running in Google Colab; upload cell skipped.")

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalized_column(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def find_symbol_csv(symbol: str) -> Path:
    pattern = re.compile(rf"^{re.escape(symbol)}(?:$|[_\-.])", re.IGNORECASE)
    matches = [path for path in CSV_DIR.glob("*.csv") if pattern.search(path.stem)]
    if not matches:
        raise FileNotFoundError(f"No CSV found for {symbol}")
    if len(matches) > 1:
        raise ValueError(f"Multiple CSV files found for {symbol}: {[path.name for path in matches]}")
    return matches[0]


def load_yahoo_csv(symbol: str) -> tuple[pd.DataFrame, dict]:
    path = find_symbol_csv(symbol)
    frame = pd.read_csv(path)
    frame.columns = [normalized_column(column) for column in frame.columns]
    if not {"date", "close", "volume"}.issubset(frame.columns):
        raise ValueError(f"{path.name}: Date, Close and Volume are required")
    adjusted_column = next((column for column in ["adj_close", "adjusted_close"] if column in frame.columns), None)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    for column in ["close", "volume"] + ([adjusted_column] if adjusted_column else []):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["date", "close", "volume"]).sort_values("date")
    frame = frame.drop_duplicates("date", keep="last").set_index("date")
    frame["adj_close"] = frame[adjusted_column] if adjusted_column else frame["close"]
    frame = frame[["adj_close", "close", "volume"]]
    frame = frame.loc[(frame["close"] > 0) & (frame["volume"] >= 0)]
    if frame.empty:
        raise ValueError(f"{path.name}: no usable rows")
    return frame, {
        "symbol": symbol,
        "source_file": str(path),
        "source_sha256": sha256_file(path),
        "first_date": frame.index.min(),
        "last_date": frame.index.max(),
        "daily_rows": len(frame),
        "load_status": "OK",
        "error": "",
    }


market_data = {}
quality_rows = []
for symbol in required_symbols:
    try:
        frame, quality = load_yahoo_csv(symbol)
        market_data[symbol] = frame
        quality_rows.append(quality)
        print(f"OK: {symbol} ({len(frame):,} rows)")
    except Exception as exc:
        quality_rows.append({"symbol": symbol, "load_status": "MISSING", "error": str(exc)})
        print(f"MISSING: {symbol}: {exc}")

data_quality = pd.DataFrame(quality_rows)
critical_missing = {FIXED_SYMBOL, BENCHMARK_SYMBOL} - set(market_data)
if critical_missing:
    raise FileNotFoundError(f"SOXX/SPY data are required: {sorted(critical_missing)}")
print(f"Loaded {len(market_data):,} of {len(required_symbols):,} requested Yahoo histories.")
display(data_quality)

## Rebuild and rank the eligible universe every month

For holding month `m`, the code uses only observations through month-end `m-1`. An ETF must also fall within its Alpha Vantage listing lifecycle and have at least 40 of the trailing 63 trading observations. Every month is evaluated independently.

In [ ]:
def liquidity_status_at_formation(symbol: str, formation_period: pd.Period) -> dict:
    cutoff = formation_period.end_time.normalize()
    if symbol not in market_data:
        return {
            "symbol": symbol, "formation_month": str(formation_period), "formation_cutoff": cutoff,
            "max_observation_used": pd.NaT, "trailing_observation_count": 0,
            "avg_daily_share_volume": np.nan, "avg_daily_dollar_volume": np.nan,
            "has_current_formation_month_data": False, "passes_data_and_liquidity": False,
            "screen_failure_reason": "YAHOO_HISTORY_MISSING",
        }
    available = market_data[symbol].loc[lambda frame: frame.index <= cutoff].copy()
    current_month = available.loc[available.index.to_period("M") == formation_period]
    trailing = available.tail(LIQUIDITY_LOOKBACK_DAYS)
    shares = float(trailing["volume"].mean()) if not trailing.empty else np.nan
    dollars = float((trailing["close"] * trailing["volume"]).mean()) if not trailing.empty else np.nan
    enough = len(trailing) >= MIN_LIQUIDITY_DAYS
    liquid = np.isfinite(shares) and shares >= MIN_AVG_DAILY_SHARE_VOLUME
    current = not current_month.empty
    passed = bool(enough and liquid and current)
    reasons = []
    if not current: reasons.append("NO_DATA_IN_FORMATION_MONTH")
    if not enough: reasons.append("INSUFFICIENT_TRAILING_OBSERVATIONS")
    if not liquid: reasons.append("LIQUIDITY_THRESHOLD_FAILED")
    max_used = available.index.max() if not available.empty else pd.NaT
    if pd.notna(max_used) and max_used > cutoff:
        raise AssertionError(f"Future observation used for {symbol}")
    return {
        "symbol": symbol, "formation_month": str(formation_period), "formation_cutoff": cutoff,
        "max_observation_used": max_used, "trailing_observation_count": len(trailing),
        "avg_daily_share_volume": shares, "avg_daily_dollar_volume": dollars,
        "has_current_formation_month_data": current, "passes_data_and_liquidity": passed,
        "screen_failure_reason": "" if passed else " | ".join(reasons),
    }


def build_monthly_shortlists():
    detail_frames = []
    shortlist_rows = []
    for holding_month in pd.period_range(FIRST_HOLDING_MONTH, LAST_HOLDING_MONTH, freq="M"):
        formation_month = holding_month - 1
        cutoff = formation_month.end_time.normalize()
        soxx = liquidity_status_at_formation(FIXED_SYMBOL, formation_month)
        shortlist_rows.append({
            "holding_month": str(holding_month), "formation_month": str(formation_month),
            "formation_cutoff": cutoff, "sleeve": "Semiconductors", "role": "FIXED_HOLDING",
            "selected_symbols": FIXED_SYMBOL if soxx["passes_data_and_liquidity"] else "",
            "selected_count": 1 if soxx["passes_data_and_liquidity"] else 0,
            "sleeve_status": "READY" if soxx["passes_data_and_liquidity"] else "ERROR_FIXED_ETF_DATA",
        })

        for sleeve, group in research_pool.groupby("sleeve", sort=False):
            rows = []
            for record in group.itertuples(index=False):
                status = liquidity_status_at_formation(record.symbol, formation_month)
                ipo = pd.Timestamp(record.ipoDate) if pd.notna(record.ipoDate) else pd.NaT
                delist = pd.Timestamp(record.delistingDate) if pd.notna(record.delistingDate) else pd.NaT
                active_lifecycle = bool(pd.notna(ipo) and ipo <= cutoff and (pd.isna(delist) or delist > cutoff))
                rows.append({
                    "holding_month": str(holding_month), "sleeve": sleeve, "name": record.name,
                    "ipoDate": ipo, "delistingDate": delist, "active_in_listing_lifecycle": active_lifecycle,
                    "classification_review": record.classification_review, **status,
                })
            month_group = pd.DataFrame(rows)
            month_group["passes_monthly_universe"] = (
                month_group["active_in_listing_lifecycle"] & month_group["passes_data_and_liquidity"]
            )
            month_group["ranking_metric"] = RANKING_METRIC
            month_group["ranking_value"] = month_group[RANKING_METRIC]
            eligible = month_group.loc[month_group["passes_monthly_universe"]].copy()
            eligible = eligible.sort_values(["ranking_value", "symbol"], ascending=[False, True])
            eligible["liquidity_rank"] = np.arange(1, len(eligible) + 1)
            selected = eligible.head(TARGET_PER_SLEEVE)["symbol"].tolist()
            month_group = month_group.merge(
                eligible[["symbol", "liquidity_rank"]], on="symbol", how="left", validate="one_to_one"
            )
            month_group["selected_top_liquidity"] = month_group["symbol"].isin(selected)
            detail_frames.append(month_group)
            shortlist_rows.append({
                "holding_month": str(holding_month), "formation_month": str(formation_month),
                "formation_cutoff": cutoff, "sleeve": sleeve, "role": "TOP_LIQUIDITY_SHORTLIST",
                "selected_symbols": ", ".join(selected), "selected_count": len(selected),
                "eligible_research_pool_count": len(eligible),
                "sleeve_status": "READY" if MIN_PER_SLEEVE <= len(selected) <= TARGET_PER_SLEEVE else "ERROR_NEEDS_2_OR_3_ETFS",
            })
    detail = pd.concat(detail_frames, ignore_index=True)
    shortlists = pd.DataFrame(shortlist_rows)
    monthly = shortlists.assign(is_ready=shortlists["sleeve_status"].eq("READY")).groupby(
        "holding_month", as_index=False
    ).agg(ready_sleeves=("is_ready", "sum"), total_sleeves=("sleeve", "nunique"))
    monthly["all_eight_sleeves_ready"] = monthly["ready_sleeves"].eq(8) & monthly["total_sleeves"].eq(8)
    return detail, shortlists, monthly


liquidity_ranking_detail, monthly_shortlists, monthly_check = build_monthly_shortlists()
# Compare identically indexed rows only. Some unavailable tickers have NaT as
# max_observation_used, so dropping nulls from one Series alone misaligns Pandas indexes.
future_audit = liquidity_ranking_detail.loc[
    liquidity_ranking_detail["max_observation_used"].notna(),
    ["max_observation_used", "formation_cutoff"],
].copy()
if (future_audit["max_observation_used"] > future_audit["formation_cutoff"]).any():
    raise AssertionError("Future-data audit failed")
display(monthly_shortlists.head(16))
display(monthly_check.head())

## Save a new timestamped audit record

Every run creates a new folder and preserves the listing snapshots, discovery rules, Yahoo data hashes, all monthly rankings and the selected shortlists. No old result is overwritten.

In [ ]:
run_id = datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir = RESULTS_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=False)

listing_union_raw.to_csv(run_dir / "alpha_listing_union_raw.csv", index=False)
lifecycle.to_csv(run_dir / "etf_lifecycle_and_classification_audit.csv", index=False)
research_pool.to_csv(run_dir / "mechanically_discovered_sector_etf_pool.csv", index=False)
download_checklist.to_csv(run_dir / "yahoo_download_checklist.csv", index=False)
data_quality.to_csv(run_dir / "source_data_quality_and_hashes.csv", index=False)
liquidity_ranking_detail.to_csv(run_dir / "monthly_liquidity_ranking_detail.csv", index=False)
monthly_shortlists.to_csv(run_dir / "monthly_top_liquidity_shortlists.csv", index=False)
monthly_check.to_csv(run_dir / "monthly_readiness_check.csv", index=False)

conventions = pd.DataFrame({
    "Item": [
        "Formation convention", "Universe source", "Sector assignment", "Ranking metric",
        "Liquidity lookback", "Minimum observations", "Monthly selected count", "SOXX",
        "Stage 1", "Stage 2", "Residual limitation",
    ],
    "Value": [
        "Holding month m uses only data through month-end m-1",
        "Alpha Vantage start-active, end-active and end-delisted listing snapshots",
        "Mechanical product-name rules; no return data; historical-source review required",
        RANKING_METRIC, LIQUIDITY_LOOKBACK_DAYS, MIN_LIQUIDITY_DAYS,
        "Top 3, or 2 when only 2 pass", "Fixed; not ranked", "Not performed", "Not performed",
        "Free-source ticker/name changes and unavailable delisted Yahoo histories",
    ],
})

xlsx_path = run_dir / "dynamic_monthly_etf_shortlists.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    monthly_check.to_excel(writer, sheet_name="Monthly readiness", index=False)
    monthly_shortlists.to_excel(writer, sheet_name="Monthly shortlists", index=False)
    liquidity_ranking_detail.to_excel(writer, sheet_name="Liquidity ranking", index=False)
    research_pool.to_excel(writer, sheet_name="Discovered pool", index=False)
    lifecycle.to_excel(writer, sheet_name="Classification audit", index=False)
    data_quality.to_excel(writer, sheet_name="Yahoo data quality", index=False)
    conventions.to_excel(writer, sheet_name="Conventions", index=False)

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
workbook = load_workbook(xlsx_path)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    worksheet.sheet_view.showGridLines = False
    for cell in worksheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="17365D")
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    for column_cells in worksheet.columns:
        width = min(max(max(len(str(cell.value or "")) for cell in column_cells) + 2, 11), 46)
        worksheet.column_dimensions[get_column_letter(column_cells[0].column)].width = width
workbook.save(xlsx_path)

manifest = {
    "run_id": run_id,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "discovered_candidate_count": int(len(research_pool)),
    "requested_yahoo_symbol_count": int(len(required_symbols)),
    "loaded_yahoo_symbol_count": int(len(market_data)),
    "all_months_ready": bool(monthly_check["all_eight_sleeves_ready"].all()),
    "stage1_completed": False,
    "stage2_completed": False,
}
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Created: {run_dir}")
print(f"Review workbook: {xlsx_path}")

In [ ]:
# Optional Colab download: one ZIP containing the complete timestamped audit record.
archive_path = Path(shutil.make_archive(str(run_dir), "zip", root_dir=run_dir))
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print(f"Local run: results ZIP saved to {archive_path.resolve()}")

## What to review

`monthly_top_liquidity_shortlists.csv` contains the two or three most actively traded ETFs for every sleeve and every holding month. It is not yet a Stage 1 winner table or a Stage 2 portfolio.

Review `Classification audit` for false keyword matches and `Yahoo data quality` for unavailable delisted histories. A paid survivor-bias-free fund database remains the rigorous solution when exact historical product identities are required.